In [102]:
!pip install mysql-connector-python

In [54]:
import pandas as pd
import numpy as np
import mysql.connector

In [55]:
conn = mysql.connector.connect(
       host = "localhost",
       user = "root",
       port = "3309",
       password = "",
       database = "project_2"
)


In [56]:
conn.is_connected()

True

In [57]:
# create function out of it
def connect_to_db():
    return mysql.connector.connect(
       host = "localhost",
       user = "root",
       port = "3309",
       password = "",
       database = "project_2")

In [58]:
connect_to_db().is_connected()

True

In [59]:
db = connect_to_db()
cursor = db.cursor(dictionary=True)

In [60]:
cursor

In [61]:
"select count(*) as total_suppliers from suppliers"

'select count(*) as total_suppliers from suppliers'

In [62]:
cursor.execute("select count(*) as total_suppliers from suppliers")

In [63]:
row = cursor.fetchone()

In [64]:
row

{'total_suppliers': 50}

In [65]:
list(row.values())[0]

50

In [66]:
queries = {
    "TOTAL SUPPLIERS" : "select count(*) as total_suppliers from suppliers",
    "TOTAL PRODUCTS": "select count(*) as total_products from products",
    "TOTAL CATEGORIES": "select count(distinct category) as total_categories from products",
    
    "(TOTAL SALES VALUE (LAST 3 MONTHS)":"""
    select round(sum(abs(s.change_quantity)*p.price),2) as total_sales_value_in_last_3_month
    from stock_entries as s
    join products p
    on s.product_id=p.product_id
    where s.change_type = "sale"
    and
    s.entry_date >= 
    (
    select date_sub(min(entry_date),interval 3 month) from stock_entries
    )""",

    "(TOTAL RESTOCK VALUE (LAST 3 MONTHS)":

     """select round(sum(abs(s.change_quantity)*p.price),2) as total_restock_value_in_last_3_month
     from stock_entries as s
     join products p
     on s.product_id=p.product_id
     where s.change_type = "restock"
     and
     s.entry_date >= 
     (
     select date_sub(max(entry_date),interval 3 month) from stock_entries
      )""",

    ("BELOW REORDER AND NO PENDING RE-ORDERS"):
    """select count(*) from products as p where p.stock_quantity<p.reorder_level
    and product_id not in
    (
    select distinct product_id from reorders where status = "Pending"
    )"""
}


In [67]:
result = {}
for label,query in queries.items():
    cursor.execute(query)
    row=cursor.fetchone()
    result[label]= list(row.values())[0]
    

In [68]:
result

{'TOTAL SUPPLIERS': 50,
 'TOTAL PRODUCTS': 208,
 'TOTAL CATEGORIES': 5,
 '(TOTAL SALES VALUE (LAST 3 MONTHS)': 6615236.75,
 '(TOTAL RESTOCK VALUE (LAST 3 MONTHS)': 152794.27,
 'BELOW REORDER AND NO PENDING RE-ORDERS': 14}

In [72]:
def get_basic_info(cursor):
    queries = {
    "TOTAL SUPPLIERS" : "select count(*) as total_suppliers from suppliers",
    "TOTAL PRODUCTS": "select count(*) as total_products from products",
    "TOTAL CATEGORIES": "select count(distinct category) as total_categories from products",
    
    "(TOTAL SALES VALUE (LAST 3 MONTHS)":"""
    select round(sum(abs(s.change_quantity)*p.price),2) as total_sales_value_in_last_3_month
    from stock_entries as s
    join products p
    on s.product_id=p.product_id
    where s.change_type = "sale"
    and
    s.entry_date >= 
    (
    select date_sub(min(entry_date),interval 3 month) from stock_entries
    )""",

    "(TOTAL RESTOCK VALUE (LAST 3 MONTHS)":

     """select round(sum(abs(s.change_quantity)*p.price),2) as total_restock_value_in_last_3_month
     from stock_entries as s
     join products p
     on s.product_id=p.product_id
     where s.change_type = "restock"
     and
     s.entry_date >= 
     (
     select date_sub(max(entry_date),interval 3 month) from stock_entries
      )""",

    ("BELOW REORDER AND NO PENDING RE-ORDERS"):
    """select count(*) from products as p where p.stock_quantity<p.reorder_level
    and product_id not in
    (
    select distinct product_id from reorders where status = "Pending"
    )"""
}

    result = {}
    for label,query in queries.items():
      cursor.execute(query)
      row=cursor.fetchone()
      result[label]= list(row.values())[0]
    return result

In [73]:
def get_table_data(cursor):
    queries = {
        "suppliers and contact details": "select supplier_name, contact_name, email, phone from suppliers",
        "product with supplier and current stock": """
            select p.product_name, s.supplier_name, p.stock_quantity 
            from products p
            join suppliers s on p.supplier_id = s.supplier_id
            order by p.product_name asc
        """,
        "product which need to reorder again": """
            select product_id, product_name, stock_quantity, reorder_level 
            from products 
            where stock_quantity < reorder_level
        """
    }

    tables = {}
    for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()

 

In [ ]:
def get_additional_tables(cursor):
     queries = {
        "suppliers and contact details": "select supplier_name, contact_name, email, phone from suppliers",
        "product with supplier and current stock": """
            select p.product_name, s.supplier_name, p.stock_quantity 
            from products p
            join suppliers s on p.supplier_id = s.supplier_id
            order by p.product_name asc
        """,
        "product which need to reorder again": """
            select product_id, product_name, stock_quantity, reorder_level 
            from products 
            where stock_quantity < reorder_level
        """
    }

     tables = {}
     for label, query in queries.items():
        cursor.execute(query)
        tables[label] = cursor.fetchall()
     return tables
     

In [ ]:
get_additional_tables(cursor)

In [ ]:
def add_new_manual_id (cursor,db,p_name,p_category,p_price,p_stock,p_reorder,p_supplier):
    proc_call = "call AddNewProductManualID(%s, %s, %s, %s, %s, %s)"
    params = (p_name,p_category,p_price,p_stock,p_reorder,p_supplier)
    cursor.execute(proc_call,params)
    db.commit()

In [ ]:
def get_categories(cursor):
    cursor.execute("select distinct category from products order by category asc")
    rows = cursor.fetchall()
    return [row["category"] for row in rows]
    

In [ ]:
get_categories(cursor)

In [ ]:
def get_suppliers(cursor):
    cursor.execute("select supplier_id,supplier_name from suppliers order by supplier_name asc")
    return cursor.fetchall()

In [ ]:
suppliers = get_suppliers(cursor)

In [ ]:
suppliers_id = [s["supplier_id"] for s in suppliers]
suppliers_name = [s["supplier_name"] for s in suppliers]

            

In [ ]:
suppliers_id

In [ ]:

suppliers_name













In [ ]:
print(lambda x: suppliers_name[suppliers_id.index(x)])

In [ ]:
def get_all_products(cursor):
   cursor.execute("select product_id,product_name from products order by product_name")
   return cursor.fetchall()

In [ ]:
def get_product_history(cursor,product_id):
   query = "select * from product_inventory_history where product_id = %s order by record_date desc"
   cursor.execute(query,(product_id,))
   return cursor.fetchall()


In [ ]:
get_all_products(cursor)

In [ ]:
def get_all_products(cursor):
   cursor.execute("select product_id, product_name from products order by  product_name")
   return cursor.fetchall()

In [ ]:
get_all_products(cursor)

In [ ]:
def get_pending_reorders(cursor):
    cursor.execute(""" 
    select r.reorder_id,p.product_name
    from reorders as r join products as p 
    on r.product_id = p.product_id
    """)
    return cursor.fetchall()


In [ ]:
get_pending_reorders(cursor)